In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BRAIN FIRST MODEL TUNING TOOLKIT — Keystroke DL Final                  ║
# ║                                                                          ║
# ║  2 DL models: CNN-LSTM and TCN. Transformer removed — too few sessions. ║
# ║                                                                          ║
# ║  4-class output (deployment target):                                     ║
# ║    positive (calm+happy) | angry | sad | neutral                         ║
# ║                                                                          ║
# ║  WHY 4-class:                                                            ║
# ║    calm+happy → indistinguishable in typing (both relaxed, low errors,  ║
# ║      moderate speed). Confirmed: calm=0.224, happy=0.340 precision.     ║
# ║    angry stays separate → fast bursts, high backspace, irregular rhythm  ║
# ║      LLM needs: calm tone, validate frustration, brief responses        ║
# ║    sad stays separate   → slow hesitant typing, long pauses             ║
# ║      LLM needs: warm empathetic tone, gentle pacing, check in           ║
# ║                                                                          ║
# ║  Everything applied:                                                     ║
# ║    ✅ One-hot key-type (12 features) — no false ordinal relationship     ║
# ║    ✅ Stat features (mean/std/median) injected at classifier head        ║
# ║    ✅ Masked attention in CNN-LSTM — padding excluded before softmax     ║
# ║    ✅ Masked avg pooling in TCN    — padding excluded from pool          ║
# ║    ✅ Padding-aware normalisation  — mean/std over real keys only        ║
# ║    ✅ Per-user calibration         — deviations from personal baseline   ║
# ║    ✅ Jitter augmentation          — ±5ms noise, real keys only          ║
# ║    ✅ 3 window sizes (25/40/60, stride=5) — covers chat msg lengths      ║
# ║    ✅ WeightedRandomSampler only   — sampler handles imbalance           ║
# ║    ✅ Unweighted loss              — no double-correction                ║
# ║    ✅ LR warmup (5ep) + cosine decay                                    ║
# ║    ✅ F1-based early stopping                                            ║
# ║    ✅ Per-fold per-class F1 logging                                      ║
# ║    ✅ Session length diagnostic                                          ║
# ║    ✅ Group-aware monitor split    — no user leaks train→monitor         ║
# ║    ✅ 30-epoch final retrain + 5% monitor split                          ║
# ║    ✅ CUDA-safe LSTM               — two 1-layer LSTMs                  ║
# ║    ✅ ONNX export (opset 13) + onnxscript installed                     ║
# ║    ✅ build_inference_sequence     — stats loaded once, correct 3-return ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# FIX 1: onnxscript added — required by PyTorch 2.x for ONNX export
import subprocess, sys
for pkg in ['onnx', 'onnxruntime', 'onnxscript']:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q', pkg])
        print(f"Installed {pkg}")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════
# SET THIS
# ══════════════════════════════════════════════════════════════════════════
DS_PATH = "/kaggle/input/datasets/ananyaaaa5/emosurv-ds"

# ══════════════════════════════════════════════════════════════════════════
# SAFE DEVICE DETECTION
# ══════════════════════════════════════════════════════════════════════════
def get_safe_device():
    if not torch.cuda.is_available():
        print("No GPU → CPU"); return torch.device('cpu')
    try:
        nn.LSTM(4,4,batch_first=True).cuda()(torch.zeros(1,5,4).cuda())
        print(f"GPU OK: {torch.cuda.get_device_name(0)}")
        return torch.device('cuda')
    except Exception as e:
        print(f"GPU failed ({type(e).__name__}) → CPU fallback")
        print("  Kaggle fix: Runtime → Change runtime type → T4 GPU")
        return torch.device('cpu')

DEVICE = get_safe_device()

# ══════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ══════════════════════════════════════════════════════════════════════════
SEED          = 42
MAX_SEQ_LEN   = 100
N_CLASSES     = 4
SEQ_FEAT_DIM  = 12      # 7 timing + 4 key_type_onehot + 1 textType_flag
STAT_DIM      = 21      # mean+std+median for 7 timing cols
JITTER_STD    = 5.0
WARMUP_EPOCHS = 5
WINDOW_CONFIGS = [(25, 5), (40, 5), (60, 5)]

TIMING_COLS = ['D1U1','D1U2','D1D2','U1D2','U1U2','D1U3','D1D3']
EMOTION_MAP = {'H':'happy','S':'sad','A':'angry','C':'calm','N':'neutral'}
FOURCLASS_MAP = {
    'happy':'positive', 'calm':'positive',
    'angry':'angry',    'sad':'sad',    'neutral':'neutral'
}

np.random.seed(SEED)
torch.manual_seed(SEED)


# ══════════════════════════════════════════════════════════════════════════
# KEY-TYPE ONE-HOT
# 4 meaningful categories instead of raw keyCode (100+ sparse values).
# One-hot, not scalar/3 — no false ordinal relationship between categories.
#   0 = alpha (a-z)     regular typing flow
#   1 = digit (0-9)     slightly different rhythm
#   2 = space/punct     natural pause boundaries
#   3 = control/bksp    editing behaviour — frustration signal for angry
# ══════════════════════════════════════════════════════════════════════════

def keycode_to_type(keycode):
    try: kc = int(float(keycode))
    except (ValueError, TypeError): return 3
    if (65<=kc<=90) or (97<=kc<=122): return 0
    if 48<=kc<=57  or 96<=kc<=105:   return 1
    if kc in (32,188,190,186,222,219,221,220,191,192,189,187): return 2
    return 3

def keycode_to_onehot(keycode):
    oh = [0.0, 0.0, 0.0, 0.0]
    oh[keycode_to_type(keycode)] = 1.0
    return oh


# ══════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD RAW DATA
# ══════════════════════════════════════════════════════════════════════════

def load_data(ds_path):
    fixed = pd.read_csv(f"{ds_path}/Fixed Text Typing Dataset.csv", sep=';')
    free  = pd.read_csv(f"{ds_path}/Free Text Typing Dataset.csv",  sep=';')
    free  = free.rename(columns={'userid':'userId'})
    fixed['textType'] = 'fixed'; free['textType'] = 'free'
    cols = ['userId','emotionIndex','keyCode','keyDown','keyUp'] + TIMING_COLS + ['textType']
    df   = pd.concat([fixed[cols], free[cols]], ignore_index=True)
    for c in TIMING_COLS + ['keyDown','keyUp','keyCode']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    print(f"Loaded {len(df):,} keystroke events | {df['userId'].nunique()} users")
    return df

typing_df = load_data(DS_PATH)


# ══════════════════════════════════════════════════════════════════════════
# STEP 2 — SESSION INDEX + 4-CLASS LABELS
# ══════════════════════════════════════════════════════════════════════════

def build_session_index(df):
    rows = []
    for (uid, emo, txt), _ in df.groupby(['userId','emotionIndex','textType']):
        emo_str = EMOTION_MAP.get(emo)
        if emo_str:
            rows.append({'userId':uid,'emotionIndex':emo,'textType':txt,
                         'emotion5':emo_str,'emotion4':FOURCLASS_MAP[emo_str]})
    return pd.DataFrame(rows).reset_index(drop=True)

sessions = build_session_index(typing_df)
le4      = LabelEncoder()
sessions['label'] = le4.fit_transform(sessions['emotion4'])

print(f"\nSessions: {len(sessions)} | 4-classes: {list(le4.classes_)}")
dist4 = {c:int(n) for c,n in zip(le4.classes_, np.bincount(sessions['label']))}
print(f"Distribution: {dist4}")


# ══════════════════════════════════════════════════════════════════════════
# STEP 3 — PER-USER CALIBRATION
# Subtract personal timing baseline → features become deviations not absolutes.
# User A types at 120ms normally, User B at 80ms.
# Both at 150ms when angry. Model learns "30ms above your norm" not "150ms".
# ══════════════════════════════════════════════════════════════════════════

def compute_user_baselines(df):
    baselines = {}
    for uid, grp in df.groupby('userId'):
        means = [float(pd.to_numeric(grp[col], errors='coerce').dropna().mean() or 0.0)
                 for col in TIMING_COLS]
        baselines[uid] = np.array(means, dtype=np.float32)
    return baselines

print("\nComputing per-user baselines...")
user_baselines = compute_user_baselines(typing_df)
print(f"  Baselines for {len(user_baselines)} users")


# ══════════════════════════════════════════════════════════════════════════
# STEP 4 — BUILD CALIBRATED SEQUENCES (12 features, one-hot key-type)
# ══════════════════════════════════════════════════════════════════════════

def session_to_seq(grp, txt_flag, uid, max_len=MAX_SEQ_LEN):
    n = len(grp)
    feats = []
    # 7 calibrated timing features
    for i, col in enumerate(TIMING_COLS):
        vals = pd.to_numeric(grp[col], errors='coerce').fillna(0).values.astype(np.float32)
        vals = vals - user_baselines[uid][i]
        feats.append(vals)
    # 4 one-hot key-type features
    kt_oh = np.array(grp['keyCode'].apply(keycode_to_onehot).tolist(), dtype=np.float32)
    for dim in range(4):
        feats.append(kt_oh[:, dim])
    # 1 textType flag
    feats.append(np.full(n, float(txt_flag), dtype=np.float32))

    seq  = np.nan_to_num(np.clip(np.stack(feats, axis=1), -500, 500))
    mask = np.ones(max_len, dtype=bool)
    if n >= max_len:
        seq = seq[:max_len]
    else:
        seq  = np.vstack([seq, np.zeros((max_len-n, SEQ_FEAT_DIM), np.float32)])
        mask[n:] = False
    return seq, mask

print("Building calibrated sequences (one-hot key-type, 12 features)...")
sequences, masks_list, labels, groups = [], [], [], []
for _, row in sessions.iterrows():
    uid = row['userId']; emo = row['emotionIndex']; txt = row['textType']
    grp = typing_df[
        (typing_df['userId']==uid) &
        (typing_df['emotionIndex']==emo) &
        (typing_df['textType']==txt)
    ]
    txt_flag = 1 if txt == 'free' else 0
    if len(grp) == 0:
        seq  = np.zeros((MAX_SEQ_LEN, SEQ_FEAT_DIM), np.float32)
        mask = np.zeros(MAX_SEQ_LEN, dtype=bool)
    else:
        seq, mask = session_to_seq(grp, txt_flag, uid)
    sequences.append(seq); masks_list.append(mask)
    labels.append(row['label']); groups.append(uid)

X_seq = np.array(sequences,  dtype=np.float32)
M_seq = np.array(masks_list, dtype=bool)
y_seq = np.array(labels,     dtype=np.int64)
g_seq = np.array(groups)
print(f"Sequences: {X_seq.shape} | Masks: {M_seq.shape}")

# Session length diagnostic — verify which window sizes actually contribute
real_lengths = M_seq.sum(axis=1)
print(f"\n  Session length stats: median={np.median(real_lengths):.0f} "
      f"mean={np.mean(real_lengths):.1f} "
      f"min={np.min(real_lengths)} max={np.max(real_lengths)}")
for w, _ in WINDOW_CONFIGS:
    pct = (real_lengths > w).mean() * 100
    print(f"  Sessions > {w} keys: {pct:.1f}% → window={w} contributes to these")


# ══════════════════════════════════════════════════════════════════════════
# STEP 4b — SESSION-LEVEL STAT FEATURES
# mean + std + median of timing cols per session.
# These are what GB uses — giving DL models access to them at the head
# bridges the gap between DL's temporal patterns and GB's stability.
# ══════════════════════════════════════════════════════════════════════════

def compute_stat_features(X, M):
    """21 = mean(7) + std(7) + median(7) of timing cols, real keys only."""
    N    = X.shape[0]
    stat = np.zeros((N, STAT_DIM), dtype=np.float32)
    for i in range(N):
        real = X[i][M[i], :7]
        if len(real) == 0: continue
        stat[i, :7]    = real.mean(0)
        stat[i, 7:14]  = real.std(0)
        stat[i, 14:21] = np.median(real, axis=0)
    return stat

print("\nComputing session-level stat features...")
S_seq = compute_stat_features(X_seq, M_seq)
print(f"  Stat features: {S_seq.shape}")


# ══════════════════════════════════════════════════════════════════════════
# STEP 5 — MULTI-WINDOW AUGMENTATION
# 3 window sizes × stride=5 → ~6000+ samples from ~388 sessions
# Stat features recomputed per window — reflect that window's actual stats
# ══════════════════════════════════════════════════════════════════════════

def augment_windows(X, M, y, g, S, configs=WINDOW_CONFIGS):
    aX, aM, ay, ag, aS = list(X), list(M), list(y), list(g), list(S)
    for seq, mask, lbl, grp, stat in zip(X, M, y, g, S):
        n_real = int(mask.sum())
        for window, stride in configs:
            if n_real <= window: continue
            for start in range(0, n_real - window, stride):
                w   = seq[start:start+window]
                wm  = np.zeros(MAX_SEQ_LEN, dtype=bool); wm[:window] = True
                pad = np.zeros((MAX_SEQ_LEN-window, SEQ_FEAT_DIM), np.float32)
                # Recompute stat for this specific window
                ws = np.zeros(STAT_DIM, dtype=np.float32)
                ws[:7]    = w[:, :7].mean(0)
                ws[7:14]  = w[:, :7].std(0)
                ws[14:21] = np.median(w[:, :7], axis=0)
                aX.append(np.vstack([w, pad])); aM.append(wm)
                ay.append(lbl); ag.append(grp); aS.append(ws)
    return (np.array(aX, np.float32), np.array(aM, bool),
            np.array(ay, np.int64),   np.array(ag),
            np.array(aS, np.float32))

print(f"\nAugmenting (windows={[w for w,s in WINDOW_CONFIGS]}, stride=5)...")
X_aug, M_aug, y_aug, g_aug, S_aug = augment_windows(X_seq, M_seq, y_seq, g_seq, S_seq)
print(f"  {len(X_seq)} sessions → {len(X_aug)} samples")
dist_aug = {c:int(n) for c,n in zip(le4.classes_, np.bincount(y_aug))}
print(f"  Augmented distribution: {dist_aug}")


# ══════════════════════════════════════════════════════════════════════════
# PADDING-AWARE NORMALISATION
# Compute mean/std ONLY over real (non-padded) keystroke rows.
# Old bug: X.reshape(-1,12).mean() included zero-padding → biased stats.
# ══════════════════════════════════════════════════════════════════════════

def compute_masked_stats(X, M):
    real = X[M]
    return real.mean(0).astype(np.float32), (real.std(0)+1e-8).astype(np.float32)

print("\nComputing padding-aware normalisation...")
GLOBAL_MEAN, GLOBAL_STD = compute_masked_stats(X_aug, M_aug)
print(f"  Mean[:3]={GLOBAL_MEAN[:3].round(2)}  (near 0 expected — calibration centres them)")
print(f"  Std[:3] ={GLOBAL_STD[:3].round(2)}")


# ══════════════════════════════════════════════════════════════════════════
# DATASET + BALANCED SAMPLER
# WeightedRandomSampler balances batches by class.
# Loss is UNWEIGHTED — sampler already handles imbalance.
# Using both would double-correct and over-penalise majority class.
# Jitter applies noise only to real positions, timing cols only.
# ══════════════════════════════════════════════════════════════════════════

class KeystrokeDataset(Dataset):
    def __init__(self, X, M, y, S, seq_mean, seq_std, stat_mean, stat_std, jitter=False):
        X_norm    = X.copy()
        X_norm[M] = (X[M] - seq_mean) / (seq_std + 1e-8)
        S_norm    = (S - stat_mean)    / (stat_std + 1e-8)
        self.X      = torch.tensor(X_norm, dtype=torch.float32)
        self.M      = torch.tensor(M,      dtype=torch.bool)
        self.y      = torch.tensor(y,      dtype=torch.long)
        self.S      = torch.tensor(S_norm, dtype=torch.float32)
        self.jitter = jitter
        self.std7   = float(seq_std[:7].mean())

    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.jitter:
            real = self.M[i]
            x[real, :7] += torch.randn(int(real.sum()), 7) * (JITTER_STD / max(self.std7, 1.0))
        return x, self.M[i], self.y[i], self.S[i]

def balanced_loader(ds, batch_size=32):
    labels  = ds.y.numpy()
    counts  = np.bincount(labels, minlength=N_CLASSES)
    weights = torch.tensor(1.0 / counts[labels], dtype=torch.float32)
    sampler = WeightedRandomSampler(weights, num_samples=len(labels), replacement=True)
    return DataLoader(ds, batch_size=batch_size, sampler=sampler)


# ══════════════════════════════════════════════════════════════════════════
# TRAINING UTILITY
# Unweighted loss (sampler handles imbalance — no double-correction).
# LR warmup: 5 epochs linear ramp → prevents large early gradients.
# F1-based early stopping — correct for imbalanced classes.
# ══════════════════════════════════════════════════════════════════════════

def train_model(model, tr_ld, val_ld, epochs=60, lr=5e-4, patience=12):
    crit   = nn.CrossEntropyLoss()   # unweighted — sampler already balances
    opt    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    warmup = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP_EPOCHS)
    cosine = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs-WARMUP_EPOCHS, 1))
    sch    = optim.lr_scheduler.SequentialLR(opt, [warmup, cosine], milestones=[WARMUP_EPOCHS])

    best_f1, best_acc, best_state, no_imp = 0.0, 0.0, None, 0
    model.to(DEVICE)

    for ep in range(epochs):
        model.train()
        for xb, mb, yb, sb in tr_ld:
            xb, mb, yb, sb = xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE), sb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb, mb, sb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, mb, yb, sb in val_ld:
                preds.extend(model(xb.to(DEVICE), mb.to(DEVICE), sb.to(DEVICE)).argmax(1).cpu().numpy())
                trues.extend(yb.numpy())

        val_f1  = f1_score(trues, preds, average='weighted', zero_division=0)
        val_acc = accuracy_score(trues, preds)
        if val_f1 > best_f1:
            best_f1=val_f1; best_acc=val_acc
            best_state={k:v.clone() for k,v in model.state_dict().items()}
            no_imp=0
        else:
            no_imp += 1
        if (ep+1) % 15 == 0:
            print(f"      ep {ep+1:3d} | f1={val_f1:.3f} | acc={val_acc:.3f} | best_f1={best_f1:.3f}")
        if no_imp >= patience:
            print(f"      Early stop ep {ep+1} (best f1={best_f1:.3f})"); break

    return best_f1, best_acc, best_state


def cross_val(model_fn, X, M, y, g, S, name, lr, epochs=60, patience=12):
    gkf = GroupKFold(n_splits=5)
    accs, f1s, all_p, all_t = [], [], [], []
    for fold, (tr, te) in enumerate(gkf.split(X, y, g)):
        print(f"  [{name}] Fold {fold+1}/5")
        seq_mean, seq_std = compute_masked_stats(X[tr], M[tr])
        stat_mean = S[tr].mean(0).astype(np.float32)
        stat_std  = (S[tr].std(0)+1e-8).astype(np.float32)

        tr_ds  = KeystrokeDataset(X[tr],M[tr],y[tr],S[tr],seq_mean,seq_std,stat_mean,stat_std,jitter=True)
        val_ds = KeystrokeDataset(X[te],M[te],y[te],S[te],seq_mean,seq_std,stat_mean,stat_std,jitter=False)
        tr_ld  = balanced_loader(tr_ds)
        val_ld = DataLoader(val_ds, batch_size=32, shuffle=False)

        m = model_fn()
        _, _, best = train_model(m, tr_ld, val_ld, epochs=epochs, lr=lr, patience=patience)
        m.load_state_dict(best); m.eval()

        preds, trues = [], []
        with torch.no_grad():
            for xb, mb, yb, sb in val_ld:
                preds.extend(m(xb.to(DEVICE),mb.to(DEVICE),sb.to(DEVICE)).argmax(1).cpu().numpy())
                trues.extend(yb.numpy())

        acc = accuracy_score(trues, preds)
        f1  = f1_score(trues, preds, average='weighted', zero_division=0)
        accs.append(acc); f1s.append(f1); all_p.extend(preds); all_t.extend(trues)
        print(f"    → acc={acc:.3f}  f1={f1:.3f}")
        # Per-class F1 per fold — shows which emotions fail on which users
        pcf = f1_score(trues, preds, average=None, zero_division=0)
        for ci, cn in enumerate(le4.classes_):
            print(f"      {cn:<10}: f1={pcf[ci]:.3f}")

    return np.mean(accs), np.std(accs), np.mean(f1s), all_t, all_p


# ══════════════════════════════════════════════════════════════════════════
# MODEL 1 — CNN-LSTM with MASKED Attention + Stat Features
#
# CNN  → local burst/pause patterns (3-5 keystroke windows)
# LSTM → how rhythm evolves over the session
# Attn → which moments carry the emotion signal
#        MASKED: padded positions get -inf before softmax so attention
#        weight never lands on padding (critical with MaxPool1d halving T)
# Stat → 21 session-level features concatenated at the head
# ══════════════════════════════════════════════════════════════════════════

class CNN_LSTM(nn.Module):
    def __init__(self, feat_dim=SEQ_FEAT_DIM, stat_dim=STAT_DIM,
                 n_classes=N_CLASSES, drop=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(feat_dim,64,3,padding=1), nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(drop*0.5),
            nn.Conv1d(64,128,5,padding=2),      nn.BatchNorm1d(128), nn.GELU(),
            nn.MaxPool1d(2), nn.Dropout(drop*0.5),
            nn.Conv1d(128,128,3,padding=1),     nn.BatchNorm1d(128), nn.GELU(),
        )
        # Two 1-layer LSTMs — avoids cudnn multi-layer dropout kernel crash
        self.lstm1 = nn.LSTM(128,128,num_layers=1,batch_first=True,bidirectional=True)
        self.drop  = nn.Dropout(drop)
        self.lstm2 = nn.LSTM(256,128,num_layers=1,batch_first=True,bidirectional=True)
        self.attn  = nn.Sequential(nn.Linear(256,64), nn.Tanh(), nn.Linear(64,1))
        self.head  = nn.Sequential(
            nn.Linear(256+stat_dim, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Dropout(drop), nn.Linear(128, n_classes)
        )

    def forward(self, x, mask=None, stat=None):
        out = self.cnn(x.permute(0,2,1)).permute(0,2,1)       # (B, T/2, 128)
        out,_ = self.lstm1(out); out = self.drop(out)
        out,_ = self.lstm2(out)                                 # (B, T/2, 256)
        scores = self.attn(out)                                 # (B, T/2, 1)
        if mask is not None:
            # Downsample mask to match post-MaxPool1d(2) length
            pool_mask = mask[:, ::2].unsqueeze(-1)              # (B, T/2, 1)
            scores = scores.masked_fill(~pool_mask, float('-inf'))
        ctx = (torch.softmax(scores, dim=1) * out).sum(1)      # (B, 256)
        if stat is not None:
            ctx = torch.cat([ctx, stat], dim=1)                 # (B, 277)
        return self.head(ctx)


# ══════════════════════════════════════════════════════════════════════════
# MODEL 2 — TCN + Stat Features
#
# Dilated convolutions see nearby AND distant keystrokes in one forward pass:
#   dilation=1 → keys 1,2,3   local burst
#   dilation=2 → keys 1,3,5   medium rhythm
#   dilation=4 → keys 1,5,9   hesitation
#   dilation=8 → keys 1,9,17  session-level rhythm
# Masked avg pool — padded positions excluded from the session summary.
# Stat features concatenated at head — same stable representations GB uses.
# ══════════════════════════════════════════════════════════════════════════

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1, drop=0.2):
        super().__init__()
        pad = (kernel-1)*dilation
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation),
            nn.BatchNorm1d(out_ch), nn.GELU(), nn.Dropout(drop),
            nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation),
            nn.BatchNorm1d(out_ch), nn.GELU(), nn.Dropout(drop),
        )
        self.res = nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else nn.Identity()
    def forward(self, x):
        return nn.functional.gelu(self.conv(x)[:,:,:x.shape[2]] + self.res(x))

class TCN(nn.Module):
    def __init__(self, feat_dim=SEQ_FEAT_DIM, stat_dim=STAT_DIM,
                 n_classes=N_CLASSES, drop=0.2):
        super().__init__()
        self.proj   = nn.Sequential(nn.Conv1d(feat_dim,64,1), nn.BatchNorm1d(64), nn.GELU())
        self.blocks = nn.Sequential(
            TCNBlock(64,128,dilation=1,drop=drop), TCNBlock(128,128,dilation=2,drop=drop),
            TCNBlock(128,128,dilation=4,drop=drop), TCNBlock(128,128,dilation=8,drop=drop),
        )
        self.head = nn.Sequential(
            nn.Linear(128+stat_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(drop), nn.Linear(64, n_classes)
        )
    def forward(self, x, mask=None, stat=None):
        out = self.proj(x.permute(0,2,1))
        out = self.blocks(out)
        if mask is not None:
            m   = mask.float().unsqueeze(1)
            out = (out*m).sum(-1) / (m.sum(-1) + 1e-8)
        else:
            out = out.mean(-1)
        if stat is not None:
            out = torch.cat([out, stat], dim=1)
        return self.head(out)


# ══════════════════════════════════════════════════════════════════════════
# TRAIN + COMPARE
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("MODEL 1 — CNN-LSTM with Masked Attention + Stat Features (4-class)")
print("="*65)
cnn_acc,cnn_std,cnn_f1,cnn_t,cnn_p = cross_val(
    CNN_LSTM, X_aug,M_aug,y_aug,g_aug,S_aug, "CNN-LSTM", lr=5e-4, epochs=60, patience=12
)

print("\n" + "="*65)
print("MODEL 2 — TCN + Stat Features (4-class)")
print("="*65)
tcn_acc,tcn_std,tcn_f1,tcn_t,tcn_p = cross_val(
    TCN, X_aug,M_aug,y_aug,g_aug,S_aug, "TCN", lr=1e-3, epochs=60, patience=12
)


# ══════════════════════════════════════════════════════════════════════════
# RESULTS
# ══════════════════════════════════════════════════════════════════════════

results = [
    ("CNN-LSTM", cnn_acc, cnn_std, cnn_f1, cnn_t, cnn_p),
    ("TCN",      tcn_acc, tcn_std, tcn_f1, tcn_t, tcn_p),
]
results_sorted = sorted(results, key=lambda x: x[3], reverse=True)
winner = results_sorted[0]

print("\n" + "="*65)
print("FINAL COMPARISON — 4-class Keystroke Emotion")
print("="*65)
print(f"  {'Model':<30} {'Acc':>7}  {'±':>5}  {'F1':>7}")
print(f"  {'-'*52}")
print(f"  {'GB baseline 5-class (Cell 4)':<30} {'0.495':>7}  {'':>5}  {'—':>7}")
for name,acc,std,f1,_,_ in results_sorted:
    marker = " ✓" if name==winner[0] else ""
    print(f"  {name+marker:<30} {acc:>7.3f}  {std:>5.3f}  {f1:>7.3f}")
print(f"  {'Chance level (4-class)':<30} {'0.250':>7}")
print(f"\n  Winner: {winner[0]}")
print(f"\n  Per-class breakdown ({winner[0]}):")
print(classification_report(winner[4], winner[5], target_names=le4.classes_, digits=3))
print("\n  LLM adapter mapping:")
print("  angry    → temperature LOW,  calm/validating tone, brief responses")
print("  sad      → temperature LOW,  warm/empathetic tone, gentle pacing")
print("  positive → temperature HIGH, energetic tone, expand topics")
print("  neutral  → temperature MID,  balanced standard response")


# ══════════════════════════════════════════════════════════════════════════
# SAVE + ONNX EXPORT
# Final retrain: 30 epochs, 5% monitoring split (group-aware), no early stop.
#
# FIX 2: GroupShuffleSplit on g_aug (user IDs) — no user appears in both
# train and monitor. Prevents augmented windows from the same session
# leaking across the split, which caused fake 0.977 monitor F1.
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print(f"SAVING {winner[0]} (deployment model)")
print("="*65)

winner_class  = CNN_LSTM if winner[0]=="CNN-LSTM" else TCN
winner_lr     = 5e-4     if winner[0]=="CNN-LSTM" else 1e-3
DEPLOY_EPOCHS = 30

final_seq_mean, final_seq_std = compute_masked_stats(X_aug, M_aug)
final_stat_mean = S_aug.mean(0).astype(np.float32)
final_stat_std  = (S_aug.std(0)+1e-8).astype(np.float32)

# FIX 2 — Group-aware monitor split
# GroupShuffleSplit ensures no user's windows appear in both train and monitor.
# Without this, same-session windows leak across the split → inflated F1.
gss = GroupShuffleSplit(n_splits=1, test_size=0.05, random_state=SEED)
tr_idx, mon_idx = next(gss.split(X_aug, y_aug, groups=g_aug))
print(f"  Train samples: {len(tr_idx)} | Monitor samples: {len(mon_idx)}")
print(f"  Monitor users: {len(np.unique(g_aug[mon_idx]))} "
      f"| Train users: {len(np.unique(g_aug[tr_idx]))}")

def make_ds(idx, jitter):
    return KeystrokeDataset(
        X_aug[idx], M_aug[idx], y_aug[idx], S_aug[idx],
        final_seq_mean, final_seq_std, final_stat_mean, final_stat_std, jitter=jitter
    )

all_ld = balanced_loader(make_ds(tr_idx, jitter=True),  batch_size=32)
mon_ld = DataLoader(make_ds(mon_idx, jitter=False), batch_size=32, shuffle=False)

deploy = winner_class().to(DEVICE)
crit_f = nn.CrossEntropyLoss()
opt    = optim.AdamW(deploy.parameters(), lr=winner_lr, weight_decay=1e-4)
warmup = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=DEPLOY_EPOCHS-WARMUP_EPOCHS)
sch    = optim.lr_scheduler.SequentialLR(opt, [warmup, cosine], milestones=[WARMUP_EPOCHS])

print(f"  Retraining {winner[0]} on 95% data ({DEPLOY_EPOCHS} epochs)...")
deploy.train()
for ep in range(DEPLOY_EPOCHS):
    for xb, mb, yb, sb in all_ld:
        xb,mb,yb,sb = xb.to(DEVICE),mb.to(DEVICE),yb.to(DEVICE),sb.to(DEVICE)
        opt.zero_grad()
        loss = crit_f(deploy(xb,mb,sb), yb)
        loss.backward(); nn.utils.clip_grad_norm_(deploy.parameters(),1.0); opt.step()
    sch.step()
    if (ep+1) % 10 == 0:
        deploy.eval()
        mp, mt = [], []
        with torch.no_grad():
            for xb,mb,yb,sb in mon_ld:
                mp.extend(deploy(xb.to(DEVICE),mb.to(DEVICE),sb.to(DEVICE)).argmax(1).cpu().numpy())
                mt.extend(yb.numpy())
        mf1 = f1_score(mt, mp, average='weighted', zero_division=0)
        # This F1 should now be close to CV F1 — not inflated
        print(f"    ep {ep+1}: monitor_f1={mf1:.3f}  (expect ~CV F1, not 0.97+)")
        deploy.train()

# Save all deployment artifacts
torch.save(deploy.state_dict(), "keystroke_deploy_model.pt")
np.save("keystroke_seq_mean.npy",   final_seq_mean)
np.save("keystroke_seq_std.npy",    final_seq_std)
np.save("keystroke_stat_mean.npy",  final_stat_mean)
np.save("keystroke_stat_std.npy",   final_stat_std)
joblib.dump(le4,            "keystroke_label_encoder.pkl")
joblib.dump(user_baselines, "keystroke_user_baselines.pkl")
joblib.dump(winner[0],      "keystroke_model_name.pkl")
print("  ✅ keystroke_deploy_model.pt")
print("  ✅ keystroke_seq_mean/std.npy  keystroke_stat_mean/std.npy")
print("  ✅ keystroke_label_encoder.pkl  (angry/neutral/positive/sad)")
print("  ✅ keystroke_user_baselines.pkl")

# ONNX export — opset 13, onnxscript installed at top
try:
    deploy.eval().cpu()
    dummy_x = torch.zeros(1, MAX_SEQ_LEN, SEQ_FEAT_DIM)
    dummy_m = torch.ones(1, MAX_SEQ_LEN, dtype=torch.bool)
    dummy_s = torch.zeros(1, STAT_DIM)
    torch.onnx.export(
        deploy, (dummy_x, dummy_m, dummy_s),
        "keystroke_deploy_model.onnx",
        input_names=["sequence","mask","stat"],
        output_names=["logits"],
        dynamic_axes={"sequence":{0:"batch"},"mask":{0:"batch"},
                      "stat":{0:"batch"},"logits":{0:"batch"}},
        opset_version=13,
    )
    print("  ✅ keystroke_deploy_model.onnx  ← use this in FastAPI")
except Exception as e:
    print(f"  ⚠ ONNX: {e}\n    Use keystroke_deploy_model.pt directly")


# ══════════════════════════════════════════════════════════════════════════
# INFERENCE HELPERS — loaded once at module level, not per call
# ══════════════════════════════════════════════════════════════════════════

_SEQ_MEAN  = np.load("keystroke_seq_mean.npy")
_SEQ_STD   = np.load("keystroke_seq_std.npy")
_STAT_MEAN = np.load("keystroke_stat_mean.npy")
_STAT_STD  = np.load("keystroke_stat_std.npy")

def build_inference_sequence(raw_events: list, uid: str) -> tuple:
    """
    raw_events: list of dicts from Chrome extension KeystrokeBuffer.
      Each dict must have: keyCode (int), D1U1,D1U2,D1D2,U1D2,U1U2,D1U3,D1D3 (ms),
                           textType ('fixed'|'free')
    uid: user ID string

    Returns three arrays:
      seq  (1, 100, 12) float32 — normalised, calibrated, one-hot key-type
      mask (1, 100)     bool    — True=real keystroke, False=padding
      stat (1, 21)      float32 — mean/std/median of timing cols
    """
    baseline = user_baselines.get(uid, np.zeros(7, dtype=np.float32))
    rows = []
    for ev in raw_events:
        row = [float(ev.get(c,0.0)) - float(baseline[i]) for i,c in enumerate(TIMING_COLS)]
        row.extend(keycode_to_onehot(ev.get('keyCode', 0)))
        row.append(1.0 if ev.get('textType','free')=='free' else 0.0)
        rows.append(row)

    seq  = np.nan_to_num(np.clip(np.array(rows, np.float32), -500, 500))
    n    = seq.shape[0]
    mask = np.zeros(MAX_SEQ_LEN, dtype=bool)
    mask[:min(n, MAX_SEQ_LEN)] = True

    if n >= MAX_SEQ_LEN: seq = seq[:MAX_SEQ_LEN]
    else: seq = np.vstack([seq, np.zeros((MAX_SEQ_LEN-n, SEQ_FEAT_DIM), np.float32)])

    real_timing = seq[mask, :7]
    stat = np.zeros(STAT_DIM, dtype=np.float32)
    if len(real_timing) > 0:
        stat[:7]    = real_timing.mean(0)
        stat[7:14]  = real_timing.std(0)
        stat[14:21] = np.median(real_timing, axis=0)

    seq[mask] = (seq[mask] - _SEQ_MEAN)  / (_SEQ_STD  + 1e-8)
    stat      = (stat       - _STAT_MEAN) / (_STAT_STD + 1e-8)
    return seq[np.newaxis], mask[np.newaxis], stat[np.newaxis]


def update_user_baseline(uid: str, raw_events: list):
    """Call after every session. Running average shifts baseline slowly."""
    vals = np.array([[float(ev.get(c,0.0)) for c in TIMING_COLS]
                     for ev in raw_events], dtype=np.float32)
    if len(vals) == 0: return
    b = user_baselines.get(uid, np.zeros(7, np.float32))
    user_baselines[uid] = 0.8*b + 0.2*vals.mean(0)
    joblib.dump(user_baselines, "keystroke_user_baselines.pkl")


print("""
─────────────────────────────────────────────────────────────
USAGE IN keystroke_engine.py
─────────────────────────────────────────────────────────────
import onnxruntime as ort
import numpy as np, joblib
from keystroke_dl_final import (
    build_inference_sequence, update_user_baseline,
    TIMING_COLS, MAX_SEQ_LEN, SEQ_FEAT_DIM, STAT_DIM
)

sess = ort.InferenceSession("keystroke_deploy_model.onnx")
le   = joblib.load("keystroke_label_encoder.pkl")
# le.classes_ → ['angry', 'neutral', 'positive', 'sad']

VA_MAP = {
    'angry':    (-0.6,  0.8),
    'neutral':  ( 0.0,  0.0),
    'positive': ( 0.7,  0.6),
    'sad':      (-0.5, -0.4),
}

def predict(raw_events, user_id):
    seq, mask, stat = build_inference_sequence(raw_events, user_id)
    logits = sess.run(["logits"],
                      {"sequence": seq, "mask": mask, "stat": stat})[0]
    e      = np.exp(logits[0] - logits[0].max())
    probs  = e / e.sum()
    emo    = le.classes_[probs.argmax()]
    v, a   = VA_MAP[emo]
    return emo, v, a, float(probs.max())

# After each session — keeps per-user calibration current
update_user_baseline(user_id, raw_events)
─────────────────────────────────────────────────────────────
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.3 MB/s eta 0:00:00
Installed onnxruntime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.6 MB/s eta 0:00:00
Installed onnxscript
GPU failed (AcceleratorError) → CPU fallback
  Kaggle fix: Runtime → Change runtime type → T4 GPU
Loaded 75,283 keystroke events | 83 users

Sessions: 388 | 4-classes: ['angry', 'neutral', 'positive', 'sad']
Distribution: {'angry': 46, 'neutral': 164, 'positive': 120, 'sad': 58}

Computing per-user baselines...
  Baselines for 83 users
Building calibrated sequences (one-hot key-type, 12 features)...
Sequences: (388, 100, 12) | Masks: (388, 100)

  Session length stats: median=100 mean=92.1 min=42 max=100
  Sessions > 25 keys: 100.0% → window=25 contributes to these
  Sessions > 40 keys: 100.0% → window=40 contributes to these
  Sessions > 60 keys: 89.4% → window=60 contributes to these

Computing session-lev

W0424 10:25:15.058000 23 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `CNN_LSTM([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CNN_LSTM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).
Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py"

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
  ✅ keystroke_deploy_model.onnx  ← use this in FastAPI

─────────────────────────────────────────────────────────────
USAGE IN keystroke_engine.py
─────────────────────────────────────────────────────────────
import onnxruntime as ort
import numpy as np, joblib
from keystroke_dl_final import (
    build_inference_sequence, update_user_baseline,
    TIMING_COLS, MAX_SEQ_LEN, SEQ_FEAT_DIM, STAT_DIM
)

sess = ort.InferenceSession("keystroke_deploy_model.onnx")
le   = joblib.load("keystroke_label_encoder.pkl")
# le.classes_ → ['angry', 'neutral', 'positive', 'sad']

VA_MAP = {
    'angry':    (-0.6,  0.8),
    'neutral':  ( 0.0,  0.0),
    'positive': ( 0.7,  0.6),
    'sad':      (-0.5, -0.4),
}

def predict(raw_events, user_id):
    seq, mask, stat = build_inference_sequence(raw_events, user_id)
    logits = sess.run(["logits"],
                      {"sequenc